# RealBasicVSR — Complete Zero → 100% Master

এটি নতুন Colab account/runtime-এ পুরো workflow চালানোর জন্য:
**Drive → environment → model → AI frames → Drive checkpoint → MP4 → original audio → final Drive output**

প্রতিটি frame Google Drive-এ save হবে। তাই মাঝপথে runtime বন্ধ হলেও frame পুনরায় তৈরি করার বদলে পরের run-এ missing frame থেকে কাজ চালানো যাবে.

In [ ]:
# STEP 1 — Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 2 — Project settings
PROJECT = "RealBasicVSR_Project"
DRIVE_ROOT = f"/content/drive/MyDrive/{PROJECT}"
INPUT_DIR = f"{DRIVE_ROOT}/input"
FRAMES_DIR = f"{DRIVE_ROOT}/frames"
RESULTS_DIR = f"{DRIVE_ROOT}/results"

import os
for p in [INPUT_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(p, exist_ok=True)

print(DRIVE_ROOT)

## STEP 3 — Put the input MP4 in Drive

প্রথমবার শুধু আপনার MP4 রাখুন:

`My Drive / RealBasicVSR_Project / input`

তারপর নিচের cell চালান।

In [ ]:
# STEP 3 — Select the first MP4 in the input folder
import glob, os
videos = sorted(glob.glob(INPUT_DIR + "/*.mp4") + glob.glob(INPUT_DIR + "/*.MP4"))
if not videos:
    raise FileNotFoundError("Drive input folder-এ MP4 নেই। ভিডিওটি সেখানে upload করুন।")
INPUT_VIDEO = videos[0]
print("Input:", INPUT_VIDEO)
!ffprobe -v error -select_streams v:0 -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 "$INPUT_VIDEO"

In [ ]:
# STEP 4 — Exact proven Python 3.10 environment
!sudo apt-get update -qq
!sudo apt-get install -y python3.10 python3.10-venv
!rm -rf /content/rbvsr_legacy
!python3.10 -m venv /content/rbvsr_legacy
!/content/rbvsr_legacy/bin/python -m pip install --upgrade pip==22.0.2
!/content/rbvsr_legacy/bin/python -m pip install numpy==1.26.4
!/content/rbvsr_legacy/bin/python -m pip install torch==1.13.1+cu117 torchvision==0.14.1 --extra-index-url https://download.pytorch.org/whl/cu117
!/content/rbvsr_legacy/bin/python -m pip install openmim==0.3.9
!/content/rbvsr_legacy/bin/python -m pip install mmcv-full==1.7.2
!/content/rbvsr_legacy/bin/python -m pip install numpy==1.26.4 opencv-python==4.8.1.78
!/content/rbvsr_legacy/bin/python -m pip install mmedit==0.16.1 tqdm==4.65.2

In [ ]:
# STEP 5 — Repo, config, checkpoint
!rm -rf /content/RealBasicVSR
!git clone -q https://github.com/ckkelvinchan/RealBasicVSR.git /content/RealBasicVSR
!mkdir -p /content/RealBasicVSR/configs /content/RealBasicVSR/checkpoints
!wget -q https://raw.githubusercontent.com/ckkelvinchan/RealBasicVSR/master/configs/realbasicvsr_x4.py -O /content/RealBasicVSR/configs/realbasicvsr_x4.py
!pip install -q gdown
!gdown 1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID -O /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

In [ ]:
# STEP 6 — Verify
!/content/rbvsr_legacy/bin/python - <<'PY'
import torch, torchvision, numpy, cv2, mmcv, mmedit
print("Python:", __import__("sys").version.split()[0])
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Torchvision:", torchvision.__version__)
print("NumPy:", numpy.__version__)
print("MMCV:", mmcv.__version__)
print("MMEditing:", mmedit.__version__)
print("OpenCV:", cv2.__version__)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
PY

In [ ]:
# STEP 7 — Create resumable inference script
script = r'''
import os, cv2, torch
from tqdm import tqdm
from mmcv import Config
from mmedit.models import build_model

INPUT_VIDEO=os.environ["RBVSR_INPUT"]
OUT_DIR=os.environ["RBVSR_FRAMES"]
CFG="/content/RealBasicVSR/configs/realbasicvsr_x4.py"
CKPT="/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"
os.makedirs(OUT_DIR, exist_ok=True)

cfg=Config.fromfile(CFG)
model=build_model(cfg.model, train_cfg=None, test_cfg=cfg.get("test_cfg"))
model=model.cuda().eval()
checkpoint=torch.load(CKPT,map_location="cpu")
model.load_state_dict(checkpoint.get("state_dict",checkpoint),strict=False)

cap=cv2.VideoCapture(INPUT_VIDEO)
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps=cap.get(cv2.CAP_PROP_FPS)
print(f"Total frames: {total}, FPS: {fps}")

for i in tqdm(range(total),desc="AI frames → Google Drive"):
    out_path=os.path.join(OUT_DIR,f"{i+1:05d}.png")
    if os.path.exists(out_path) and os.path.getsize(out_path)>1000:
        continue
    cap.set(cv2.CAP_PROP_POS_FRAMES,i)
    ok,frame=cap.read()
    if not ok: raise RuntimeError(f"Could not read frame {i+1}")
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    img=torch.from_numpy(rgb).float().permute(2,0,1).unsqueeze(0).unsqueeze(0).cuda()/255.0
    with torch.no_grad(): out=model(img,test_mode=True)
    result=out["output"] if isinstance(out,dict) else out
    if result.dim()==5: result=result[0,0]
    elif result.dim()==4: result=result[0]
    result=result.clamp(0,1).permute(1,2,0).cpu().numpy()
    bgr=cv2.cvtColor((result*255).round().astype("uint8"),cv2.COLOR_RGB2BGR)
    if not cv2.imwrite(out_path,bgr): raise RuntimeError(f"Failed to save {out_path}")
cap.release()
print("AI frame generation complete.")
'''
open("/content/RealBasicVSR/inference_drive_resume.py","w").write(script)
print("Inference script ready.")

In [ ]:
# STEP 8 — AI processing with Drive checkpoint/resume
import os
os.environ["RBVSR_INPUT"]=INPUT_VIDEO
os.environ["RBVSR_FRAMES"]=FRAMES_DIR
!RBVSR_INPUT="$RBVSR_INPUT" RBVSR_FRAMES="$RBVSR_FRAMES" /content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_drive_resume.py

In [ ]:
# STEP 9 — Verify all frames before encoding
import glob, os
frames=sorted(glob.glob(FRAMES_DIR+"/*.png"))
print("Frame count on Drive:",len(frames))
if not frames:
    raise RuntimeError("No PNG frames found.")
print("First:",os.path.basename(frames[0]))
print("Last :",os.path.basename(frames[-1]))

In [ ]:
# STEP 10 — Copy frames temporarily to local storage for fast FFmpeg encoding
# Drive remains the permanent checkpoint; this local copy can be recreated after a reset.
!rm -rf /content/rbvsr_frames
!mkdir -p /content/rbvsr_frames
!cp "$FRAMES_DIR"/*.png /content/rbvsr_frames/

In [ ]:
# STEP 11 — Encode the 4x video
!mkdir -p /content/rbvsr_results
!ffmpeg -y -framerate 25 -i /content/rbvsr_frames/%05d.png -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p /content/rbvsr_results/upscaled_video.mp4

In [ ]:
# STEP 12 — Add original audio
!ffmpeg -y -i /content/rbvsr_results/upscaled_video.mp4 -i "$INPUT_VIDEO" -map 0:v:0 -map 1:a:0? -c:v copy -c:a copy -shortest /content/rbvsr_results/final_4x.mp4

In [ ]:
# STEP 13 — Verify final file
!ffprobe -v error -show_entries format=duration,size -show_entries stream=index,codec_name,width,height,r_frame_rate,codec_type -of default=noprint_wrappers=1 /content/rbvsr_results/final_4x.mp4

In [ ]:
# STEP 14 — Save final MP4 to Google Drive
FINAL_DRIVE=f"{RESULTS_DIR}/final_4x.mp4"
!cp /content/rbvsr_results/final_4x.mp4 "$FINAL_DRIVE"
print("Saved:", FINAL_DRIVE)
!ls -lh "$FINAL_DRIVE"

## Recovery rule

যদি Step 8 চলাকালীন runtime disconnect/reset হয়:

- Drive-এর `frames` folder-এ যত PNG তৈরি হয়েছে সেগুলো থাকবে।
- নতুন runtime-এ Notebook আবার চালান।
- একই Drive project folder ব্যবহার করুন।
- Step 8 আবার চালালে existing PNG **skip** হবে এবং missing frame-গুলো তৈরি হবে।
- সব frame পাওয়া গেলে Step 9 থেকে আবার এগিয়ে ভিডিও তৈরি করুন।

**Frame checkpoint এবং final MP4 দুটোই Google Drive-এ থাকবে।**